# Munk Translation Pipeline
## *Guide des égarés* (Paris, 1856) — Part 1 → English

Fetches Munk's French from Sefaria's API, translates each segment via Claude, and saves a structured JSON file usable by the trifold reader.

**Run cells in order.** Progress is saved after every segment — if the notebook crashes or times out, re-run the **▶ Run Pipeline** cell and it will resume automatically from the last completed segment.

---

## Step 1 — Install dependencies
*(Run once per session)*

In [ ]:
!pip install -q google-genai requests tqdm
print('Done.')

## Step 2 — Imports

In [ ]:
from google import genai
import requests
import json, os, re, time, datetime
from getpass import getpass
from tqdm.notebook import tqdm
from IPython.display import display, HTML
print('Imports OK.')

## Step 3 — Gemini API key
Get yours at [console.anthropic.com](https://console.anthropic.com). It will not be displayed after you paste it.

In [ ]:
from google.colab import userdata
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
client = genai.Client(api_key=GOOGLE_API_KEY)
print('Client initialized.')

## Step 4 — Configuration
Edit the values in this cell if needed, then run it.

In [ ]:
# ── Model ───────────────────────────────────────────────────
# gemini-2.5-flash  →  fast, high quality, ~$4-8 for all of Part 1
# claude-opus-4-6    →  highest quality,    ~$20-30 for all of Part 1
MODEL = 'gemini-latest-flash'

# ── Output file ──────────────────────────────────
OUTPUT_FILE = '/Users/rayhabbaz/Library/CloudStorage/GoogleDrive-rhabbaz@gmail.com/My Drive/munk_part1_translations.json'

# ── Rate limiting ────────────────────────
# Seconds to wait between API calls. Increase to 2.0 if you hit errors.
DELAY = 1.0

# ── Sefaria version string for Munk's French ──────────
MUNK_VER = 'french|Guide_des_\u00e9gar\u00e9s,_trans._by_Salomon`_Munk,_Paris,_1856_[fr]'
TEXT_BASE = 'Guide_for_the_Perplexed,_Part_1'

print(f'Model: {MODEL}')
print(f'Output: {OUTPUT_FILE}')

## Step 5 — System prompt and glossary
These are pre-filled from your prompt design document. Expand the glossary here before starting a full run.

In [ ]:
SYSTEM_PROMPT = ''''ROLE AND TASK
You are translating the main body of Salomon Munk's French philosophical
translation of Maimonides' Guide for the Perplexed, published as Guide des
egarés, Paris, 1856. Munk's French is itself a translation from the original
Judeo-Arabic (Dalalat al-Ha'irin). Your sole task is to render Munk's French
into English faithfully and completely.

You are translating Munk — not Maimonides. Do not consult, harmonize with,
or import phrasing from other English translations of the Guide (Friedländer
1881, Pines 1963, or any other). If you are aware of how another translator
renders a given passage, set that knowledge aside entirely.

REGISTER
Adopt the register of serious Victorian scholarly prose: formal, precise,
somewhat elevated, but not artificially archaic. Prefer Latinate vocabulary
where Munk makes Latinate choices in French (e.g., "faculty" not "ability";
"intellect" not "mind"; "substance" not "stuff"; "apprehension" not "grasp").
Munk's sentences are often long and periodic; preserve their syntactic
structure where English permits, rather than breaking them into shorter units.
Do not modernize. Do not colloquialize.

COMPLETENESS
Translate every word Munk wrote. Do not summarize, compress, or omit any
portion of the input. Do not add explanatory glosses, parenthetical
clarifications, or interpretive expansions of your own. If you are uncertain
about a word or phrase, render your best judgment and flag it with a bracketed
note (see Translator's Notes below).

MULTILINGUAL CONTENT — CRITICAL RULES
Munk frequently embeds Hebrew, Arabic, Greek, and Latin within his French
prose. Handle each as follows:

  Hebrew in Hebrew script: preserve exactly as Munk has it, including any
  vowel pointing (niqqud) he supplies. Do not transliterate what Munk has
  written in Hebrew characters. Do not normalize or modernize the spelling.

  Munk's own transliterations of Hebrew or Arabic into Latin characters
  (e.g., "selèm", "Kalâm", "Moré Neboukhim"): preserve his transliteration
  system exactly, including all diacritics, exactly as printed. Do not
  standardize to modern academic transliteration conventions.

  Arabic in Arabic script: preserve as Munk has it.

  Latin quotations or phrases: translate into English inline, in square
  brackets immediately following the Latin, marked [Lat.: ...].

  Greek terms or phrases: preserve in Greek script as Munk has them.

  Biblical and rabbinic citations: preserve Munk's citation form. Use the
  English name of the book (e.g., "Psalms" for "Psaumes") but do not alter
  his chapter/verse numbering.

MUNK'S FOOTNOTES
Munk's edition contains extensive translator's footnotes (notes de bas de
page). These footnotes are a critical part of his scholarly apparatus and
must be preserved in full. Do not omit, summarize, or compress them.

Render each footnote using standard Markdown footnote syntax:
  - At the point in the body text where Munk places his footnote marker,
    insert a reference anchor: [^1], [^2], etc., numbered sequentially
    within each segment starting from 1.
  - At the end of the translated segment, collect all footnote bodies
    in order, each on its own line, formatted as:
    [^1]: Translated text of Munk's first footnote.
    [^2]: Translated text of Munk's second footnote.

Translate the content of the footnotes with the same fidelity, register,
and rules that govern the main body text. The same glossary, multilingual
preservation rules, and completeness requirements apply inside footnotes.
If a footnote itself contains Hebrew, Arabic, Greek, or Latin, handle it
exactly as specified in the Multilingual Content rules above.

TRANSLATOR'S NOTES
You may append a translator's note only when:
  (a) a French word or phrase is genuinely ambiguous between two readings
      with meaningfully different philosophical implications, or
  (b) a proper name, technical term, or abbreviation is unresolvable.
Format: [tr. note: ...] immediately after the relevant word or clause.
Keep notes to one sentence. Do not add notes for routine difficulties.

STRICT OUTPUT RULES
1. Provide the English translation ONLY. No French source text.
2. DO NOT "think out loud".
3. DO NOT provide a "Task", "Refining", or "Thinking" section.
4. DO NOT provide any preamble, notes, or commentary.
5. Begin immediately with the first word of the English text.

ONE-SHOT EXAMPLE:
User: Translate "Un homme de science m’a fait une objection."
Assistant: A man of science put to me an objection.'''

GLOSSARY = ''' TERMINOLOGY GLOSSARY — apply strictly throughout

CORE PHILOSOPHICAL TERMS
  intellect          → intellect          (not "mind")
  entendement        → understanding      (distinct from intellect; preserve distinction)
  forme              → form               (not "shape")
  matière            → matter             (not "material" or "stuff")
  faculté            → faculty            (not "ability" or "power")
  imagination        → imagination
  perfection         → perfection         (Aristotelian sense)
  agent              → agent
  cause efficiente   → efficient cause
  hypostase          → hypostasis         (not "substance")
  attributs          → attributes         (theological sense)
  essence            → essence
  accident           → accident           (Aristotelian category)
  substance          → substance          (Aristotelian category)
  mouvement          → motion             (not "movement"; Aristotelian)
  repos              → rest               (counterpart to mouvement)
  âme                → soul              (not "mind")
  puissance          → potentiality       (Aristotle's dunamis)
  acte               → actuality          (Aristotle's energeia)
  matière première   → prime matter
  intelligence séparée → separate intellect

MUNK'S TRANSLITERATIONS — preserve exactly, including diacritics
  Kalâm, Motécallemîn, Moré Neboukhim, selèm, El, Adonaï

PROPER NAMES — Anglicize consistently
  Aristote → Aristotle | Platon → Plato | Averroès → Averroes
  Avicenne → Avicenna | Maïmonide → Maimonides
  Ibn Tibbon → Ibn Tibbon | Ibn Ézra → Ibn Ezra

[Expand this glossary before starting the full run — see the companion .md file]'''


## Step 6 — Sefaria API utilities
Probes the Sefaria API chapter by chapter to build the complete list of segments and fetch their French text.

In [ ]:
import json

LOCAL_JSON_PATH = '/Users/rayhabbaz/Downloads/Guide for the Perplexed - en - Guide des égarés, trans. by Salomon Munk, Paris, 1856 [fr].json'

def flatten_text(t):
    '''Sefaria sometimes returns nested lists; flatten to plain string.'''
    if isinstance(t, list):
        return ' '.join(flatten_text(x) for x in t)
    return t or ''

def build_segment_list():
    '''
    Loads the ENTIRE book structure from local Sefaria JSON export.
    Returns a list of dicts: {ref, chapter, paragraph, french}.
    '''
    print('Loading text from local Sefaria JSON export...')
    with open(LOCAL_JSON_PATH, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    segments = []
    
    # 1. Letter to R Joseph
    letter = data['text'].get('Letter to R Joseph son of Judah', [])
    for i, t in enumerate(letter):
        french = flatten_text(t).strip()
        if french:
            segments.append({
                'ref': f'Guide_for_the_Perplexed,_Letter_to_R_Joseph_son_of_Judah.{i+1}',
                'chapter': 'Letter',
                'paragraph': i + 1,
                'french': french
            })
            
    # 2. Prefatory Remarks
    prefatory = data['text'].get('Prefatory Remarks', [])
    for i, t in enumerate(prefatory):
        french = flatten_text(t).strip()
        if french:
            segments.append({
                'ref': f'Guide_for_the_Perplexed,_Prefatory_Remarks.{i+1}',
                'chapter': 'Prefatory',
                'paragraph': i + 1,
                'french': french
            })
            
    # 3. Parts 1, 2, 3
    for part in ['Part 1', 'Part 2', 'Part 3']:
        part_data = data['text'].get(part, {})
        
        # Introduction for the Part
        for i, t in enumerate(part_data.get('Introduction', [])):
            french = flatten_text(t).strip()
            if french:
                segments.append({
                    'ref': f'Guide_for_the_Perplexed,_{part.replace(" ", "_")},_Introduction.{i+1}',
                    'chapter': f'{part} Intro',
                    'paragraph': i + 1,
                    'french': french
                })
                
        # Chapters for the Part
        chapters = part_data.get('', [])
        for ch_idx, chapter_paras in enumerate(chapters):
            ch = ch_idx + 1
            for i, t in enumerate(chapter_paras):
                french = flatten_text(t).strip()
                if french:
                    segments.append({
                        'ref': f'Guide_for_the_Perplexed,_{part.replace(" ", "_")}.{ch}.{i+1}',
                        'chapter': f'{part} Ch {ch}',
                        'paragraph': i + 1,
                        'french': french
                    })
                    
    print(f'\nTotal: {len(segments)} segments extracted.')
    return segments

# Run it
ALL_SEGMENTS = build_segment_list()
# Build a quick-lookup dict: ref → french text
FRENCH_BY_REF = {s['ref']: s['french'] for s in ALL_SEGMENTS}
SEG_REFS = [s['ref'] for s in ALL_SEGMENTS]  # ordered list of all refs




## Step 7 — Load or initialise results file
If `OUTPUT_FILE` already exists (from a previous run), existing translations are loaded and will be skipped during the pipeline.

In [ ]:
def load_results():
    if os.path.exists(OUTPUT_FILE):
        with open(OUTPUT_FILE, encoding='utf-8') as f:
            existing = json.load(f)
        done = len(existing.get('segments', {}))
        print(f'Resuming: {done} segments already translated.')
        return existing
    print('No existing file found — starting fresh.')
    return {
        'metadata': {
            'title': 'Guide des égarés — Part 1',
            'munk_edition': 'Paris, 1856',
            'model': MODEL,
            'part': 1,
            'created': datetime.datetime.now().isoformat(),
            'completed': False
        },
        'segments': {},
        'stats': {
            'total_segments': len(ALL_SEGMENTS),
            'completed': 0,
            'total_tokens': 0
        }
    }

def save_results(results):
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

RESULTS = load_results()

## Step 8 — Translation engine

In [ ]:
def extract_tr_notes(text):
    '''Pull out all [tr. note: ...] annotations from the translated text.'''
    return re.findall(r'\[tr\.\s*note:\s*([^\]]+)\]', text)

def translate_segment(ref, french, previous_french=''):
    '''
    Send one segment to Gemini and return a result dict.
    Raises on API error (caller handles retry logic).
    '''
    context_block = (
        previous_french if previous_french
        else '[This is the first segment — no preceding context.]'
    )
    user_msg = (
        f'CONTEXT — preceding segment (do not translate; for continuity only):
'
        f'"""
{context_block}
"""

'
        f'TRANSLATE THIS SEGMENT:
'
        f'"""
{french}
"""'
    )
    
    # Combine SYSTEM_PROMPT and GLOSSARY for the system instruction
    full_system_instruction = f"{SYSTEM_PROMPT}

{GLOSSARY}"
    
    response = client.models.generate_content(
        model=MODEL,
        contents=user_msg,
        config=genai.types.GenerateContentConfig(
            system_instruction=full_system_instruction,
            temperature=0.2,
        )
    )
    english = response.text.strip()
    in_tok = response.usage_metadata.prompt_token_count if response.usage_metadata else 0
    out_tok = response.usage_metadata.candidates_token_count if response.usage_metadata else 0
    return {
        'english': english,
        'tr_notes': extract_tr_notes(english),
        'input_tokens': in_tok,
        'output_tokens': out_tok,
        'timestamp': datetime.datetime.now().isoformat()
    }

print('Translation function ready.')


## Step 9 — Test mode *(run this before the full pipeline)*
Translates the first 3 segments so you can verify the output quality and check that the API key, Sefaria fetch, and prompts are all working correctly. Does **not** write to the output file.

Review the output carefully:
- Is the register right (formal, Latinate, Victorian scholarly)?
- Are Hebrew characters preserved exactly?
- Are Munk's transliterations (circumflexes, grave accents) intact?
- Are the glossary terms applied correctly?

If anything looks wrong, adjust the system prompt or glossary in Step 5 before running the full pipeline.

In [ ]:
def run_test():
    print('=== TEST MODE: first 3 segments ===\n')
    divider = '─' * 65
    for i, seg in enumerate(ALL_SEGMENTS[:3]):
        prev = ALL_SEGMENTS[i-1]['french'] if i > 0 else ''
        print(f'Translating {seg["ref"]}...')
        result = translate_segment(seg['ref'], seg['french'], prev)
        print(divider)
        print(f'SEGMENT: {seg["ref"]}')
        print(f'FRENCH (first 300 chars):\n  {seg["french"][:300]}')
        print(f'ENGLISH (first 300 chars):\n  {result["english"][:300]}')
        if result['tr_notes']:
            print(f'TRANSLATOR NOTES: {result["tr_notes"]}')
        print(f'Tokens: {result["input_tokens"]} in / {result["output_tokens"]} out')
        print(divider + '\n')
        time.sleep(DELAY)
    print('Test complete. Proceed to the pipeline cell if output looks correct.')

run_test()

## Step 10 — ▶ Run the full pipeline
Translates all segments, skipping any already in the output file. Safe to interrupt and re-run — progress is saved after every segment.

**Estimated time:** ~15–40 minutes for all of Part 1, depending on model and segment count.

In [ ]:
def run_pipeline():
    global RESULTS
    RESULTS = load_results()  # reload in case of restart
    done_refs = set(RESULTS['segments'].keys())
    todo = [s for s in ALL_SEGMENTS if s['ref'] not in done_refs]

    if not todo:
        print('All segments already translated. Nothing to do.')
        return

    print(f'Segments to translate : {len(todo)}')
    print(f'Already completed     : {len(done_refs)}')
    print(f'Total in Part 1       : {len(ALL_SEGMENTS)}')
    print(f'Model                 : {MODEL}
')

    RESULTS['stats']['total_segments'] = len(ALL_SEGMENTS)

    # Status bar showing exactly how many JSON pieces are left to process
    pbar = tqdm(todo, desc=f'Remaining pieces: {len(todo)}', unit='seg')
    for i, seg in enumerate(pbar):
        ref = seg['ref']
        remaining = len(todo) - i
        pbar.set_description(f'Remaining pieces: {remaining}')
        pbar.set_postfix_str(ref.split(".")[-2] + "." + ref.split(".")[-1])

        # Get the previous segment's French for context
        idx = SEG_REFS.index(ref)
        prev_french = FRENCH_BY_REF.get(SEG_REFS[idx - 1], '') if idx > 0 else ''

        # Translate with retry on rate-limit
        for attempt in range(3):
            try:
                result = translate_segment(ref, seg['french'], prev_french)
                break
            except Exception as e:
                if '429' in str(e):
                    wait = 60 * (attempt + 1)
                    tqdm.write(f'Rate limit hit — waiting {wait}s...')
                    time.sleep(wait)
                else:
                    tqdm.write(f'Error at {ref}: {e}')
                    save_results(RESULTS)
                    raise

        RESULTS['segments'][ref] = {
            'ref': ref,
            'chapter': seg['chapter'],
            'paragraph': seg['paragraph'],
            'french': seg['french'],
            **result
        }
        RESULTS['stats']['completed'] = len(RESULTS['segments'])
        RESULTS['stats']['total_tokens'] += (
            result['input_tokens'] + result['output_tokens']
        )
        save_results(RESULTS)  # write after every segment
        time.sleep(DELAY)

    RESULTS['metadata']['completed'] = True
    RESULTS['metadata']['finished'] = datetime.datetime.now().isoformat()
    save_results(RESULTS)

    total_tok = RESULTS['stats']['total_tokens']
    flash_cost = total_tok * 0.075 / 1_000_000
    print(f'
✓ Complete.')
    print(f'  Segments translated : {len(RESULTS["segments"])}')
    print(f'  Total tokens used   : {total_tok:,}')
    print(f'  Estimated cost (Flash pricing): ~')


## Step 11 — Review progress and sample output
Run this at any point to check how far the pipeline has gotten and inspect a random segment.

In [ ]:
import random

def show_stats():
    if not os.path.exists(OUTPUT_FILE):
        print('No output file yet — run the pipeline first.')
        return
    with open(OUTPUT_FILE, encoding='utf-8') as f:
        data = json.load(f)
    stats = data['stats']
    segs = data['segments']
    meta = data['metadata']
    print(f'Progress   : {stats["completed"]}/{stats["total_segments"]} segments')
    pct = 100 * stats['completed'] / max(stats['total_segments'], 1)
    bar = '█' * int(pct // 5) + '░' * (20 - int(pct // 5))
    print(f'           : [{bar}] {pct:.1f}%')
    print(f'Tokens     : {stats["total_tokens"]:,}')
    est = stats['total_tokens'] * 0.075 / 1_000_000
    print(f'Est. cost  : ~${est:.2f} (Flash pricing)')
    print(f'Model      : {meta["model"]}')
    print(f'Completed  : {meta.get("completed", False)}')
    if segs:
        notes_count = sum(len(s.get('tr_notes', [])) for s in segs.values())
        print(f'Tr. notes  : {notes_count} logged')
        print()
        sample_ref = random.choice(list(segs.keys()))
        sample = segs[sample_ref]
        print(f'─── Random sample: {sample_ref} ───')
        print(f'FRENCH  : {sample["french"][:250]}...')
        print(f'ENGLISH : {sample["english"][:250]}...')
        if sample.get('tr_notes'):
            print(f'TR.NOTES: {sample["tr_notes"]}')

show_stats()

## Step 12 — Export translator's notes
Collects all `[tr. note: ...]` flags from the translation into a separate file for review. Check these before treating the translation as complete.

In [ ]:
def export_notes():
    if not os.path.exists(OUTPUT_FILE):
        print('No output file yet.')
        return
    with open(OUTPUT_FILE, encoding='utf-8') as f:
        data = json.load(f)
    notes_file = OUTPUT_FILE.replace('.json', '_translator_notes.json')
    notes = {}
    for ref, seg in data['segments'].items():
        if seg.get('tr_notes'):
            notes[ref] = {
                'chapter': seg['chapter'],
                'paragraph': seg['paragraph'],
                'notes': seg['tr_notes'],
                'french_context': seg['french'][:200],
                'english_context': seg['english'][:200]
            }
    with open(notes_file, 'w', encoding='utf-8') as f:
        json.dump(notes, f, ensure_ascii=False, indent=2)
    print(f'Exported {len(notes)} notes to {notes_file}')
    return notes_file

export_notes()

## Step 13 — Download output files
Downloads both the translation JSON and the translator's notes file to your local machine.

In [ ]:
from google.colab import files as colab_files

notes_file = OUTPUT_FILE.replace('.json', '_translator_notes.json')

if os.path.exists(OUTPUT_FILE):
    print(f'Downloading {OUTPUT_FILE}...')
    colab_files.download(OUTPUT_FILE)
else:
    print(f'Main output file not found: {OUTPUT_FILE}')

if os.path.exists(notes_file):
    print(f'Downloading {notes_file}...')
    colab_files.download(notes_file)
else:
    print('No translator notes file yet — run Step 12 first.')